# Practical 8: Two-Stage Detector Fine-Tuning: Faster R-CNN on Crop Pests
**Course:** CSE471 — Deep Learning for Computer Vision  
**Unit:** Unit IV (Object Detection & Localization) | **CO:** CO4  
**Problem Statement Context:** SIH26131 — High-Precision Bounding-Box Detection for Crop Damage  

---

## 1. Two-Stage Detector Architecture
Faster R-CNN (Ren et al., 2015) decouples the detection pipeline into:
1. **Backbone + Feature Pyramid Network (FPN):** Extracts multi-scale visual features.
2. **Region Proposal Network (RPN):** Slides across feature maps to generate class-agnostic regions of interest (RoIs) using anchor boxes.
3. **RoIAlign:** Accurately extracts fixed-size feature vectors from non-integer proposed coordinates without quantization artifacts.
4. **Fast R-CNN Heads:** Parallel branches computing softmax class probabilities and bounding box regression offsets:

$$\mathcal{L}_{total} = \mathcal{L}_{rpn\_cls} + \mathcal{L}_{rpn\_reg} + \mathcal{L}_{roi\_cls} + \mathcal{L}_{roi\_reg}$$

In [ ]:
import sys
from pathlib import Path
import numpy as np
import torch
import matplotlib.pyplot as plt

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))

from src.dataset import CropDetectionDataset, detection_collate_fn
from src.detector import build_crop_faster_rcnn, compute_ap

device = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
print(f"[✓] Active Device: {device}")

## 2. Load Crop Pest Detection Dataset

In [ ]:
det_img_dir = REPO_ROOT / "data" / "samples" / "detection" / "images"
det_xml_dir = REPO_ROOT / "data" / "samples" / "detection" / "annotations_voc"

classes = ["aphid", "armyworm", "whitefly", "leaf_miner", "blast_lesion", "blight_spot"]
dataset = CropDetectionDataset(
    images_dir=det_img_dir,
    annotations_dir=det_xml_dir,
    classes=classes,
    target_size=(416, 416)
)

print(f"[✓] Loaded {len(dataset)} annotated detection samples.")
sample_img, sample_target = dataset[0]
print(f"Sample tensor shape : {sample_img.shape}")
print(f"Annotated boxes count: {len(sample_target['boxes'])}")

## 3. Instantiate Detector & Forward Step

In [ ]:
model = build_crop_faster_rcnn(num_classes=len(classes) + 1).to(device)

# Forward pass with dummy image batch
dummy_batch = [sample_img.to(device)]
dummy_targets = [{k: v.to(device) for k, v in sample_target.items()}]

model.train()
try:
    loss_dict = model(dummy_batch, dummy_targets)
    print("Faster R-CNN Multi-Task Loss Components:")
    for k, v in loss_dict.items():
        print(f"  {k}: {v.item():.4f}")
except Exception as e:
    # Fallback if torchvision detection model is SimpleCropDetector
    box_preds, cls_preds = model(dummy_batch[0].unsqueeze(0))
    print(f"[✓] Fallback detector forward success: box shape {box_preds.shape}, cls shape {cls_preds.shape}")

## 4. Detection Metric: Precision-Recall Curve & Average Precision (mAP)

In [ ]:
# Synthetic PR-curve simulation for mAP@0.5 calculation
recalls = np.linspace(0.0, 1.0, 20)
precisions = np.array([1.0, 0.98, 0.95, 0.93, 0.90, 0.88, 0.85, 0.82, 0.80, 0.76, 0.72, 0.68, 0.63, 0.58, 0.51, 0.44, 0.35, 0.25, 0.15, 0.05])

ap_50 = compute_ap(recalls, precisions)
print(f"[✓] Computed Average Precision (AP@0.5): {ap_50 * 100:.2f}%")

plt.figure(figsize=(7, 5))
plt.plot(recalls, precisions, label=f"PR Curve (AP = {ap_50:.2f})", color="navy", lw=2)
plt.fill_between(recalls, precisions, alpha=0.2, color="navy")
plt.title("Precision-Recall Curve (Foliar Pest Detection)")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.ylim([0.0, 1.05])
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 5. Academic Comparison: Two-Stage vs. Single-Stage
- **Faster R-CNN (Two-Stage):** Higher localization accuracy on tiny objects (individual aphids, young leaf lesions) due to explicit RoIAlign, but slower inference (~15 FPS).
- **YOLO (Single-Stage):** Extreme speed (>45 FPS), ideal for drone video streams and edge farm monitoring, with slightly lower sensitivity for clustered microscopic pests.